In [1]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result 
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing

[SETUP] Project Root: /home/bia/Documents/AutoDDG-Enhanced
[SETUP] Cache Directory: /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/profile_cache


In [18]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# --- Experiment Config ---
# --- Experiment Config ---
DATABASE_PATH_ = '../src/autoddg/database.json'
RESULTS_FILE_RELATIVE = 'autoddg_experiment_results.csv' # Keep the relative part
PROFILE_CACHE_DIR = 'profile_cache'

# Define necessary base directories
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))

# --- NEW: Define the ABSOLUTE path for the results file ---
# We assume the results file path is relative to the directory where the current script is.
RESULTS_FILE_ABSOLUTE = os.path.join(script_dir, RESULTS_FILE_RELATIVE)

# Rename the variable used in your logging
RESULTS_FILE = RESULTS_FILE_ABSOLUTE
# DATABASE_PATH_ = '../src/autoddg/database.json'  # Ensure this path is correct\n",
# RESULTS_FILE = 'prompt-experiments/autoddg_experiment_results.csv'
# PROFILE_CACHE_DIR = 'profile_cache' # Directory to save/load profiles\n",

# script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
# DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
# PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))
# ABSOLUTE_CACHE_DIR = os.path.join(script_dir, PROFILE_CACHE_DIR)


# --- Define Evaluation Class ---
class Eval(BaseEvaluator):
    def __init__(self, model_name: str = MODEL_CONFIG["model_name"]):
        client = OpenAI(
            api_key=MODEL_CONFIG["api_key"], 
            base_url=MODEL_CONFIG["base_url"]
        )
        super().__init__(client=client, model_name=model_name)

# Initialize Core Tools
client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])
auto_ddg = AutoDDG(client=client, model_name=MODEL_CONFIG["model_name"])
auto_ddg.set_evaluator(Eval())

In [3]:
# Assuming DATABASE_PATH, run_with_caching, and auto_ddg are already defined

# 1. Load the database (adjust DATABASE_PATH if needed)
with open(DATABASE_PATH, 'r') as f:
    raw_database = json.load(f)
    # Convert keys to integers if they are dataset IDs
    database = {int(k): v for k, v in raw_database.items()}

# 2. Get the first dataset entry
try:
    # Use next(iter()) to reliably get the first key/value pair.
    # The key is the dataset_id, and the value (the info dictionary) is dataset_info.
    dataset_id, dataset_info = next(iter(database.items()))
except StopIteration:
    print("Error: The database file is empty.")
    exit()

# dataset_info is now the full metadata dictionary (e.g., {'dataset_name': '...', 'description': '...'}).

print(f"Running experiment on first dataset: ID='{dataset_id}'")

# 3. Call the run_with_caching function
run_with_caching(dataset_id, dataset_info, auto_ddg)

Running experiment on first dataset: ID='3222451'

[RUNNER] Attempting to process The FluPRINT database...
[CACHE] Loaded profiles for 3222451 from cache.
[RUNNER] Successfully retrieved profiles for The FluPRINT database.
  Topic: It appears to be a dataset related to "Immune Cell Analysis" or "Cytokine Profiling".
--------------------------------------------------


In [9]:
# Assuming 'database' is the dictionary loaded from your database.json file

dataset_id = 5569235
# Use .get() for safe lookup, which returns None if the key is not found
dataset_info = database.get(dataset_id) 

if dataset_info is None:
    print(f"Error: Dataset ID '{dataset_id}' not found in the database.")
    exit()

# If the ID is found, you can now proceed to call your runner function
print(f"Running experiment on selected dataset: ID='{dataset_id}', Name='{dataset_info['dataset_name']}'")

# 3. Call the run_with_caching function
run_with_caching(dataset_id, dataset_info, auto_ddg)

Running experiment on selected dataset: ID='5569235', Name='The Global Carbon Project's fossil CO2 emissions dataset'

[RUNNER] Attempting to process The Global Carbon Project's fossil CO2 emissions dataset...
--- Loading Data and Running Core Profiling for The Global Carbon Project's fossil CO2 emissions dataset (Generating) ---
  Attempting to load data from: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/data/5569235.csv


/home/bia/miniconda3/envs/nlp-final/lib/python3.11/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


[CACHE] Saved profiles for 5569235 to /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/profile_cache/5569235_profiles.pkl
[RUNNER] Successfully retrieved profiles for The Global Carbon Project's fossil CO2 emissions dataset.
  Topic: Carbon Emissions Data
--------------------------------------------------


In [9]:
print(dataset_id, dataset_info)

3222451 {'dataset_name': 'The FluPRINT database', 'dataset_path': 'src/autoddg/related/data/fluprint_export.csv', 'related_paper_path': 'src/autoddg/related/papers/3222451.pdf', 'description': 'The FluPRINT represents fully integrated and normalized immunology measurements from eight clinical studies taken from 740 individuals undergoing influenza vaccination with inactivated or live attenuated seasonal influenza vaccines from 2007 to 2015 at the Stanford Human Immune Monitoring Center. The FluPRINT dataset contains information on more than 3,000 parameters measured using mass cytometry, flow cytometry, phosphorylation-specific cytometry, multiplex cytokine assays, clinical lab tests (hormones and complete blood count), serological profiling and virological tests. In the dataset, vaccine protection is measured using a hemagglutination inhibition (HAI) assay, and following FDA guidelines individuals are marked as high or low responders depending on the HAI antibody titers after vaccinat

In [3]:
database = None
try:
    # Load the database JSON
    with open(DATABASE_PATH, 'r') as f:
        database = json.load(f)
        print(database)
except FileNotFoundError:
    print(f"ERROR: Database file not found at {DATABASE_PATH}")
except json.JSONDecodeError:
    print(f"ERROR: Could not decode JSON from {DATABASE_PATH}. Is the file correctly formatted?")
except Exception as e:
    print(f"An unexpected error occurred while loading the database: {e}")

# Iterate through the database entries
for dataset_id, dataset_info in database.items():
    try:
        # This calls the function that implements the cache-first logic
        run_with_caching(dataset_id, dataset_info, auto_ddg) 
    except Exception as e:
        print(f"\n{'#'*50}")
        print(f"FATAL ERROR: FAILED ON DATASET ID {dataset_id} ({dataset_info.get('dataset_name', 'Unknown')})")
        print(f"Error: {e}")
        print(f"{'#'*50}\n")
        # Continue to the next dataset instead of stopping the whole loop
        continue

print("\n=======================================================")
print("ALL CACHING RUNS FINISHED.")
print("=======================================================")


{'3222451': {'dataset_name': 'The FluPRINT database', 'dataset_path': 'src/autoddg/related/data/fluprint_export.csv', 'related_paper_path': 'src/autoddg/related/papers/3222451.pdf', 'description': 'The FluPRINT represents fully integrated and normalized immunology measurements from eight clinical studies taken from 740 individuals undergoing influenza vaccination with inactivated or live attenuated seasonal influenza vaccines from 2007 to 2015 at the Stanford Human Immune Monitoring Center. The FluPRINT dataset contains information on more than 3,000 parameters measured using mass cytometry, flow cytometry, phosphorylation-specific cytometry, multiplex cytokine assays, clinical lab tests (hormones and complete blood count), serological profiling and virological tests. In the dataset, vaccine protection is measured using a hemagglutination inhibition (HAI) assay, and following FDA guidelines individuals are marked as high or low responders depending on the HAI antibody titers after vacc

/home/bia/miniconda3/envs/nlp-final/lib/python3.11/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)
Unmatched latitude columns: ['LATS2', 'LATS1', 'DLAT', 'LCLAT1', 'MALAT1', 'LAT', 'LAT2', 'PLAT']
Unmatched longitude columns: ['TXLNG', 'LONRF3', 'LONRF2', 'LONRF1', 'LONP2', 'IGLON5', 'LONP1']



##################################################
FATAL ERROR: FAILED ON DATASET ID 14052302 (GRPM Dataset)
Error: 'dict' object has no attribute 'lower'
##################################################


[RUNNER] Attempting to process Variation in Leaf Reflectance Spectra Across the California Flora Partitioned by Evolutionary History, Geographic Origin, and Deep Time...
--- Loading Data and Running Core Profiling for Variation in Leaf Reflectance Spectra Across the California Flora Partitioned by Evolutionary History, Geographic Origin, and Deep Time (Generating) ---
  Attempting to load data from: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/data/Griffith-etal-2022-Tilden-ASD-spectra-corrected-means-continuum-removal.csv


/home/bia/miniconda3/envs/nlp-final/lib/python3.11/site-packages/datamart_profiler/core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


KeyboardInterrupt: 

In [13]:
test_id = '3222451'
test_profile = load_profile_from_cache(test_id)

[CACHE] Loaded profiles for 3222451 from cache.


In [13]:
# Cell 4: Define Prompts to Test (Selects from the imported dictionary)

# Define which prompts you want to run for this experiment
PROMPTS_TO_TEST = {
    "V1_Revised": ALL_RELATED_WORK_PROMPTS["V1_Revised"],
    # "V2_Aggressive": ALL_RELATED_WORK_PROMPTS["V2_Aggressive"],
    "V2_Hybrid": ALL_RELATED_WORK_PROMPTS["V2_Hybrid"]
}
print(f"Testing {len(PROMPTS_TO_TEST)} related work prompts.")



Testing 2 related work prompts.


In [14]:
dataset_id = 7651129
# Use .get() for safe lookup, which returns None if the key is not found
dataset_info = database.get(dataset_id) 

if dataset_info is None:
    print(f"Error: Dataset ID '{dataset_id}' not found in the database.")
    exit()

# If the ID is found, you can now proceed to call your runner function
print(f"Running experiment on selected dataset: ID='{dataset_id}', Name='{dataset_info['dataset_name']}'")

DATASET_NAME = dataset_info['dataset_name'] 

# 2. Define the PAPER_FILE path
# Use the correct key: 'related_paper_path'
PAPER_FILE_RELATIVE = dataset_info['related_paper_path'] 

# Resolve the absolute path needed by auto_ddg.analyze_related()
# Note: PROJECT_ROOT must be defined earlier in your script.
PAPER_FILE = os.path.join(PROJECT_ROOT, PAPER_FILE_RELATIVE) 

profiles = load_profile_from_cache(dataset_id=dataset_id)
# --- Unpack the Profiles (Assuming 'profiles' dict is loaded from cache) ---
basic_profile = profiles["basic_profile"]
semantic_profile = profiles["semantic_profile"]
data_topic = profiles["data_topic"]
dataset_sample = profiles["dataset_sample"] 

print(f"Experiment setup complete for: {DATASET_NAME}")
print(f"PDF location set to: {PAPER_FILE}")

Running experiment on selected dataset: ID='7651129', Name='Coronavirus disease (COVID-19) case data - South Africa'
[CACHE] Loaded profiles for 7651129 from cache.
Experiment setup complete for: Coronavirus disease (COVID-19) case data - South Africa
PDF location set to: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/elife-78933-v1.pdf


In [ ]:
# -----------------------------------------------------------------
## 1. Baseline (Vanilla AutoDDG) Description
# -----------------------------------------------------------------

print("\n--- Running Baseline (Vanilla) Description ---")

# Generates description using only dataset features
prompt_baseline, description_baseline = auto_ddg.describe_dataset(
    dataset_sample=dataset_sample,
    dataset_profile=basic_profile,
    use_profile=True,
    semantic_profile=semantic_profile,
    use_semantic_profile=True,
    data_topic=data_topic,
    use_topic=True,
    use_related_profile=False  # **Vanilla: Only uses internal dataset profiles**
)

baseline_scores = auto_ddg.evaluate_description(description_baseline)
print(f"Baseline Scores: {baseline_scores}")

# Log result
log_result(
    prompt_name="N/A", 
    description_type="Vanilla_AutoDDG", 
    description=description_baseline, 
    raw_scores=baseline_scores,
    dataset_name=DATASET_NAME,
    file_path=RESULTS_FILE
    # related_profile is omitted
)

print("-" * 50)

# -----------------------------------------------------------------
## 2. Augmented (AutoDDG + Related Work) Description
# -----------------------------------------------------------------

print("\n--- Running Augmented (AutoDDG + Related Work) Description ---")


for prompt_name, extraction_prompt in PROMPTS_TO_TEST.items():
    print(f"\n--- Running Augmented Test with Prompt: {prompt_name} ---")
    
    # Step A: Analyze related work using the current prompt
    related_profile = auto_ddg.analyze_related(
        pdf_path=PAPER_FILE,
        dataset_name=DATASET_NAME,
        extraction_prompt=extraction_prompt,
        max_pages=10
    )
    print(f"Related Work Summary: {related_profile['summary'][:150]}...")

    # Step B: Generate description with the new related profile
    prompt_augmented, description_augmented = auto_ddg.describe_dataset(
        dataset_sample=dataset_sample,
        dataset_profile=basic_profile,
        use_profile=True,
        semantic_profile=semantic_profile,
        use_semantic_profile=True,
        data_topic=data_topic,
        use_topic=True,
        related_profile=related_profile,
        use_related_profile=True # Augmented
    )
    
    # Step C: Evaluate and Log
    augmented_scores = auto_ddg.evaluate_description(description_augmented)
    print(f"Augmented Scores ({prompt_name}): {augmented_scores}")
    
    log_result(
        prompt_name=prompt_name, 
        description_type="Augmented_AutoDDG", 
        description=description_augmented, 
        raw_scores=augmented_scores,
        dataset_name=DATASET_NAME,
        file_path=RESULTS_FILE,
        related_profile=related_profile # Now logs the profile!
    )
    
print("\nAll experiments complete. Results saved to:", RESULTS_FILE)


--- Running Baseline (Vanilla) Description ---
Baseline Scores: Completeness: 8
Conciseness: 6
Readability: 9
Logged Vanilla_AutoDDG with Prompt N/A to /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/autoddg_experiment_results.csv
--------------------------------------------------

--- Running Augmented (AutoDDG + Related Work) Description ---

--- Running Augmented Test with Prompt: V1_Revised ---
Reading PDF from: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/elife-78933-v1.pdf
Successfully extracted text from 10 pages (total: 50 pages)
Total characters extracted: 37033
Extracting related work profile for dataset: Coronavirus disease (COVID-19) case data - South Africa
Sending 37837 characters to LLM...
Successfully extracted profile (2257 characters)
Related Work Summary: This is a research paper on the COVID-19 pandemic in South Africa, specifically focusing on the dynamics of three major SARS-CoV-2 variants: Beta, Del...
Augmented Scores (V1_Revised): Co

In [7]:
# Cell 6: Run and Log Augmented Descriptions for Multiple Prompts

for prompt_name, extraction_prompt in PROMPTS_TO_TEST.items():
    print(f"\n--- Running Augmented Test with Prompt: {prompt_name} ---")
    
    # Step A: Analyze related work using the current prompt
    related_profile = auto_ddg.analyze_related(
        pdf_path=PAPER_FILE,
        dataset_name=DATASET_NAME,
        extraction_prompt=extraction_prompt,
        max_pages=10
    )
    print(f"Related Work Summary: {related_profile['summary'][:150]}...")

    # Step B: Generate description with the new related profile
    prompt_augmented, description_augmented = auto_ddg.describe_dataset(
        dataset_sample=dataset_sample,
        dataset_profile=basic_profile,
        use_profile=True,
        semantic_profile=semantic_profile,
        use_semantic_profile=True,
        data_topic=data_topic,
        use_topic=True,
        related_profile=related_profile,
        use_related_profile=True # Augmented
    )
    
    # Step C: Evaluate and Log
    augmented_scores = auto_ddg.evaluate_description(description_augmented)
    print(f"Augmented Scores ({prompt_name}): {augmented_scores}")
    
    log_result(
        prompt_name=prompt_name, 
        description_type="Augmented_AutoDDG", 
        description=description_augmented, 
        raw_scores=augmented_scores,
        dataset_name=DATASET_NAME,
        file_path=RESULTS_FILE,
        related_profile=related_profile # Now logs the profile!
    )
    
print("\nAll experiments complete. Results saved to:", RESULTS_FILE)

Ignoring wrong pointing object 43 0 (offset 0)



--- Running Augmented Test with Prompt: V1_Revised ---
Reading PDF from: ../src/autoddg/related/papers/code15.pdf
Successfully extracted text from 10 pages (total: 10 pages)
Total characters extracted: 55673
Extracting related work profile for dataset: CODE-15%: a large scale annotated dataset of 12-lead ECGs
Sending 56479 characters to LLM...
Successfully extracted profile (689 characters)
Related Work Summary: The dataset search engine has extracted the following factual context:

* The study found that deep learning models can accurately predict atrial fibr...


Ignoring wrong pointing object 43 0 (offset 0)


Augmented Scores (V1_Revised): Here are my scores based on the Evaluation Criteria:

Evaluation Form (scores ONLY):

Completeness: 9
Conciseness: 8
Readability: 9
Logged Augmented_AutoDDG with Prompt V1_Revised to autoddg_experiment_results.csv

--- Running Augmented Test with Prompt: V2_Hybrid ---
Reading PDF from: ../src/autoddg/related/papers/code15.pdf
Successfully extracted text from 10 pages (total: 10 pages)
Total characters extracted: 55673
Extracting related work profile for dataset: CODE-15%: a large scale annotated dataset of 12-lead ECGs
Sending 56621 characters to LLM...
Successfully extracted profile (2333 characters)
Related Work Summary: **Dataset Entry:**

**Title:** Comparative Analysis of Deep Learning Models for Electrocardiogram (ECG) Signal Processing and Arrhythmia Detection

**...
Augmented Scores (V2_Hybrid): Here are my scores based on the Evaluation Criteria:

Completeness: 9
Conciseness: 8
Readability: 9
Logged Augmented_AutoDDG with Prompt V2_Hybrid to auto

In [ ]:

import json
import os

def run_experiment(dataset):
    
    
    dataset_name = dataset["dataset_name"]
    data_file = dataset["dataset_path"]
    paper_file = dataset.get("related_paper_path")
        
    print("--- Loading Data and Running Core Profiling ---")
    #make sure size is appropriate
    if os.path.getsize(data_file) > 10 * 1024 * 1024:  # 10 MB size limit
        #sample 
        df = pd.read_csv(data_file, nrows=100000)  
        
    else: 
        df = pd.read_csv(data_file)
    sample_df, dataset_sample = get_sample(df, sample_size=100)

    basic_profile, structural_profile = auto_ddg.profile_dataframe(df)
    semantic_profile = auto_ddg.analyze_semantics(sample_df)
    data_topic = auto_ddg.generate_topic(DATASET_NAME, None, dataset_sample)

    print("Profiling Complete.")
    
        
    print("\n--- Running Baseline (Vanilla) Test ---")
    prompt_baseline, description_baseline = auto_ddg.describe_dataset(
            dataset_sample=dataset_sample,
            dataset_profile=basic_profile,
            use_profile=True,
            semantic_profile=semantic_profile,
            use_semantic_profile=True,
            data_topic=data_topic,
            use_topic=True,
            use_related_profile=False  # Vanilla
        )

    baseline_scores = auto_ddg.evaluate_description(description_baseline)
    print(f"Baseline Scores: {baseline_scores}")

    # Log result using the new utility function (passing required file/dataset args)
    log_result(
        prompt_name="N/A", 
        description_type="Vanilla_AutoDDG", 
        description=description_baseline, 
        raw_scores=baseline_scores,
        dataset_name=DATASET_NAME,
        file_path=RESULTS_FILE
        # related_profile is omitted (defaults to None)
    )
    
    PROMPTS_TO_TEST = {
        "V1_Revised": ALL_RELATED_WORK_PROMPTS["V1_Revised"],
        "V2_Hybrid": ALL_RELATED_WORK_PROMPTS["V2_Hybrid"]
    }
    
        
    for prompt_name, extraction_prompt in PROMPTS_TO_TEST.items():
        print(f"\n--- Running Augmented Test with Prompt: {prompt_name} ---")
        
        # Step A: Analyze related work using the current prompt
        related_profile = auto_ddg.analyze_related(
            pdf_path=paper_file,
            dataset_name=dataset_name,
            extraction_prompt=extraction_prompt,
            max_pages=10
        )
        print(f"Related Work Summary: {related_profile['summary'][:150]}...")

        # Step B: Generate description with the new related profile
        prompt_augmented, description_augmented = auto_ddg.describe_dataset(
            dataset_sample=dataset_sample,
            dataset_profile=basic_profile,
            use_profile=True,
            semantic_profile=semantic_profile,
            use_semantic_profile=True,
            data_topic=data_topic,
            use_topic=True,
            related_profile=related_profile,
            use_related_profile=True # Augmented
        )
        
        # Step C: Evaluate and Log
        augmented_scores = auto_ddg.evaluate_description(description_augmented)
        print(f"Augmented Scores ({prompt_name}): {augmented_scores}")
        
        log_result(
            prompt_name=prompt_name, 
            description_type="Augmented_AutoDDG", 
            description=description_augmented, 
            raw_scores=augmented_scores,
            dataset_name=DATASET_NAME,
            file_path=RESULTS_FILE,
            related_profile=related_profile # Now logs the profile!
        )
        
    print("\nDataset {dataset} completed.--> Results saved to:", RESULTS_FILE)


In [ ]:
       
    
###Runnning all datasets in the database

#Load the dabase 
with open("src/autoddg/database.json") as f:
    database = json.load(f)



for dataset_id, dataset in database.items():
    try:
        run_experiment(dataset)
    except Exception as e:
        print(f"FAILED ON {dataset_id}: {e}")
        continue

In [ ]:


#LOOP THROUGH ALL DATASETS IN DATABASE

# --- Experiment Config ---
RESULTS_FILE = "autoddg_experiment_results.csv"

# --- Load Database ---
with open("src/autoddg/database.json") as f:
    database = json.load(f)

# --- Iterate through datasets ---
for dataset_id, dataset in database.items():
    DATASET_NAME = dataset["dataset_name"]
    DATA_FILE = dataset["dataset_path"]
    PAPER_FILE = dataset.get("related_paper_path")

    print("Running experiment on:", DATASET_NAME)
    print("Data file:", DATA_FILE)
    print("Paper file:", PAPER_FILE)
    print("----")
    
    # Run Evaluation
    print("--- Loading Data and Running Core Profiling ---")
    
    #some datasets are too large to load fully and will have to be sampled 
    if os.path.getsize(DATA_FILE) > 100000000:  # 100 MB size limit for full load
        print("Dataset too large, loading a sample for profiling.")
        df = pd.read_csv(DATA_FILE, nrows=10000)  # Load only first 10,000 rows for sampling
    else: 
        df = pd.read_csv(DATA_FILE)
        
    sample_df, dataset_sample = get_sample(df, sample_size=100)

    basic_profile, structural_profile = auto_ddg.profile_dataframe(df)
    semantic_profile = auto_ddg.analyze_semantics(sample_df)
    data_topic = auto_ddg.generate_topic(DATASET_NAME, None, dataset_sample)

    print("Profiling Complete.")
    